# Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datasets import load_dataset, load_from_disk, Dataset, DatasetDict
import json
from tqdm import tqdm
from datatrove.pipeline.readers import ParquetReader
from transformers import AutoTokenizer
from pathlib import Path
from collections import defaultdict
from datatrove.pipeline.readers import ParquetReader

In [ ]:
# Defining the names of all languages

african_language_list = [
    'aeb_Arab',
    'afr_Latn',
    'aka_Latn',
    'amh_Ethi',
    'ary_Arab',
    'arz_Arab',
    'bam_Latn',
    'bem_Latn',
    'cjk_Latn',
    'dik_Latn',
    'dyu_Latn',
    'ewe_Latn',
    'fon_Latn',
    'fuv_Latn',
    'gaz_Latn',
    'hau_Latn',
    'ibo_Latn',
    'kab_Latn',
    'kam_Latn',
    'kbp_Latn',
    'kea_Latn',
    'kik_Latn',
    'kin_Latn',
    'kmb_Latn',
    'knc_Arab',
    'knc_Latn',
    'kon_Latn',
    'lin_Latn',
    'lua_Latn',
    'lug_Latn',
    'luo_Latn',
    'mos_Latn',
    'nqo_Nkoo',
    'nso_Latn',
    'nus_Latn',
    'nya_Latn',
    'plt_Latn',
    'run_Latn',
    'sag_Latn',
    'sna_Latn',
    'som_Latn',
    'sot_Latn',
    'ssw_Latn',
    'swh_Latn',
    'taq_Latn',
    'taq_Tfng',
    'tir_Ethi',
    'tsn_Latn',
    'tso_Latn',
    'tum_Latn',
    'twi_Latn',
    'tzm_Tfng',
    'umb_Latn',
    'wol_Latn',
    'xho_Latn',
    'yor_Latn',
    'zul_Latn',
]

high_lang_list = {
    'eng_Latn',
    'fra_Latn',
    'por_Latn',
    'arb_Arab'
    }

In [ ]:
# Dealing with Wura Dataset mapping

# Wura dataset follows the flores 200 mapping
flores_200_mapping = {
'afr_Latn': 'afr',
'amh_Ethi': 'amh',
'arb_Arab': 'ara',
'asm_Beng': 'asm',
'ast_Latn': 'ast',
'azj_Latn': 'azj',
'arz_Arab': 'arz',
'bel_Cyrl': 'bel',
'ben_Beng': 'ben',
'bos_Latn': 'bos',
'bul_Cyrl': 'bul',
'cat_Latn': 'cat',
'ceb_Latn': 'ceb',
'ces_Latn': 'ces',
'ckb_Arab': 'ckb',
'cym_Latn': 'cym',
'dan_Latn': 'dan',
'deu_Latn': 'deu',
'ell_Grek': 'ell',
'eng_Latn': 'eng',
'est_Latn': 'est',
'fin_Latn': 'fin',
'fra_Latn': 'fra',
'fuv_Latn': 'ful',
'gaz_Latn': 'gaz',
'gle_Latn': 'gle',
'glg_Latn': 'glg',
'guj_Gujr': 'guj',
'hau_Latn': 'hau',
'heb_Hebr': 'heb',
'hin_Deva': 'hin',
'hrv_Latn': 'hrv',
'hun_Latn': 'hun',
'hye_Armn': 'hye',
'ibo_Latn': 'ibo',
'ind_Latn': 'ind',
'isl_Latn': 'isl',
'ita_Latn': 'ita',
'jav_Latn': 'jav',
'jpn_Jpan': 'jpn',
'kam_Latn': 'kam',
'kan_Knda': 'kan',
'kat_Geor': 'kat',
'kaz_Cyrl': 'kaz',
'khm_Khmr': 'khm',
'kir_Cyrl': 'kir',
'kin_Latn': 'kin',
'kor_Hang': 'kor',
'lao_Laoo': 'lao',
'lij_Latn': 'Latvian',
'lim_Latn': 'kea',
'lin_Latn': 'lin',
'lit_Latn': 'lit',
'ltz_Latn': 'ltz',
'lug_Latn': 'lug',
'luo_Latn': 'luo',
'lvs_Latn': 'lav',
'mal_Mlym': 'mal',
'mar_Deva': 'mar',
'mkd_Cyrl': 'mkd',
'mlt_Latn': 'mlt',
'khk_Cyrl': 'mon',
'mri_Latn': 'mri',
'mya_Mymr': 'mya',
'nld_Latn': 'nld',
'nob_Latn': 'nob',
'npi_Deva': 'npi',
'nso_Latn': 'nso',
'nya_Latn': 'nya',
'oci_Latn': 'oci',
'gaz_Latn': 'orm',
'ory_Orya': 'ory',
'pan_Guru': 'pan',
'pes_Arab': 'fas',
'pol_Latn': 'pol',
'por_Latn': 'por',
'pbt_Arab': 'pus',
'plt_Latn': 'plt',
'ron_Latn': 'ron',
'rus_Cyrl': 'rus',
'slk_Latn': 'slk',
'sna_Latn': 'sna',
'snd_Arab': 'snd',
'som_Latn': 'som',
'spa_Latn': 'spa',
'srp_Cyrl': 'srp',
'swc_Latn': 'swc',
'swe_Latn': 'swe',
'swh_Latn': 'swa',
'tam_Taml': 'tam',
'tel_Telu': 'tel',
'tgk_Cyrl': 'tgk',
'tir_Ethi': 'tir',
'tgl_Latn': 'tgl',
'tha_Thai': 'tha',
'tur_Latn': 'tur',
'ukr_Cyrl': 'ukr',
'umb_Latn': 'umb',
'urd_Arab': 'urd',
'uzn_Latn': 'uzb',
'vie_Latn': 'vie',
'wol_Latn': 'wol',
'xho_Latn': 'xho',
'yor_Latn': 'yor',
'zho_Hans': 'zho_simpl',
'zho_Hant': 'zho_trad',
'zsm_Latn': 'msa',
'zul_Latn': 'zul'}


# List of all wura languages,
wura_langs = [
    "afr", "amh", "arz", "eng", "fra", "hau", "ibo", "kin",
    "mlg", "nya", "orm", "por", "sna", "som", "sot",
    "swa", "tir", "xho", "yor", "zul"
]


In [ ]:
# Defining the madlad 400 mapping

afri_madlad_langs = {
    "afr_Latn": "af",
    "aka_Latn": "ak",
    "amh_Ethi": "am",
    "bam_Latn": "bm",
    "dik_Latn": "din",
    "dyu_Latn": "dyu",
    "ewe_Latn": "ee",
    "fon_Latn": "fon",
    "fuv_Latn": "ff",
    "gaz_Latn": "om",   
    "hau_Latn": "ha",  
    "ibo_Latn": "ig",   
    "kbp_Latn": "kbp",
    "kin_Latn": "rw",
    "kmb_Latn": "kmb",
    "kon_Latn": "kg",
    "lin_Latn": "ln",
    "lug_Latn": "lg",
    "run_Latn": "rn",
    "sag_Latn": "sg",
    "sna_Latn": "sn",
    "som_Latn": "so",
    "sot_Latn": "st",
    "ssw_Latn": "ss",
    "swh_Latn": "sw",
    "tir_Ethi": "ti",
    "tsn_Latn": "tn",
    "tso_Latn": "ts",
    "tzm_Tfng": "ber",
    "wol_Latn": "wo",
    "xho_Latn": "xh",
    "yor_Latn": "yo",
    "zul_Latn": "zu"
}

hr_madlad_langs = {
    "eng_Latn": "en",
    "fra_Latn": "fr",
    "por_Latn": "pt",
    "arb_Arab": "ar"
}

In [ ]:
# MGSM like mapping


afri_mgsm_langs = {
    "amh_Ethi": "amh",
    "ewe_Latn": "ewe",
    "gaz_Latn": "orm",  
    "hau_Latn": "hau",
    "ibo_Latn": "ibo",
    "kin_Latn": "kin",
    "lin_Latn": "lin",
    "lug_Latn": "lug",
    "sna_Latn": "sna",
    "swh_Latn": "swa",
    "sot_Latn": "sot",
    "twi_Latn": "twi",
    "wol_Latn": "wol",
    "xho_Latn": "xho",
    "yor_Latn": "yor",
    "zul_Latn": "zul"
}

mgsm_langs = {
    "eng_Latn": "en",
    "fra_Latn": "fr"
}

# Data folder Creation

In [ ]:
# Create folder for each language in all languages
for lang in african_language_list:
    os.makedirs(f"data/{lang}", exist_ok=True)

for lang in high_lang_list:
    os.makedirs(f"data/{lang}", exist_ok=True)

# Fineweb 2

In [ ]:
# Inspect dataset

dataset = load_dataset(
    "parquet",
    data_files={"train": "hf://datasets/HuggingFaceFW/fineweb-2/data/por_Latn/train/000_00000.parquet"},
    split="train",
    streaming=True
)

print("Available columns:", dataset.features)


## Crafting African Language Set that is part of FW2

In [ ]:
from datatrove.pipeline.readers import ParquetReader

afri_fw2_existing_langs = []

for lang_code in african_language_list:
    path = f"hf://datasets/HuggingFaceFW/fineweb-2/data/{lang_code}/train"
    try:
        reader = ParquetReader(path, limit=1)  # Try to load just 1 file
        _ = next(reader())                     # Trigger read
        afri_fw2_existing_langs.append(lang_code)
        print(f"Exists: {lang_code}")
    except Exception as e:
        print(f" Missing: {lang_code}")

print("\nAvailable languages:", afri_fw2_existing_langs)

## Crafting High Resource Language Set that is part of FW2

In [ ]:
from datatrove.pipeline.readers import ParquetReader

hr_fw2_existing_langs = []

for lang_code in high_lang_list:
    path = f"hf://datasets/HuggingFaceFW/fineweb-2/data/{lang_code}/train"
    try:
        reader = ParquetReader(path, limit=1)  # Try to load just 1 file
        _ = next(reader())                     # Trigger read
        hr_fw2_existing_langs.append(lang_code)
        print(f"Exists: {lang_code}")
    except Exception as e:
        print(f" Missing: {lang_code}")

print("\nAvailable languages:", hr_fw2_existing_langs)

## Downloading African Language Datasets for FW2

In [ ]:
import os
import fsspec
from tqdm import tqdm

def download_fw2_parquet(lang_code: str, save_dir: str):
    """
    Download the FineWeb-2 train split for one language **unchanged**
    (same .parquet files, no decoding) and store them under `save_dir/lang_code/`.
    """
    remote = f"hf://datasets/HuggingFaceFW/fineweb-2/data/{lang_code}/train"
    fs, _ = fsspec.core.url_to_fs(remote)          # fsspec handles the HF filesystem
    os.makedirs(save_dir, exist_ok=True)

    # Download all parquet files there, because its african languages so I download all of them
    for idx, path in enumerate(tqdm(fs.glob(f"{remote}/*.parquet"),
                                    desc=f"Downloading {lang_code}"), start=1):
        fname      = f"{lang_code}_{idx:04d}_fw2.parquet"     # File name structure: afr_Latn_0001.parquet, ...
        local_path = os.path.join(save_dir, fname)
        fs.get(path, local_path)  # Download the file as is


# Loop through all languages and download the parquet files
for lang_code in afri_fw2_existing_langs:
    save_dir = f"data/{lang_code}"
    download_fw2_parquet(lang_code, save_dir)


## Downloading first 1BT for High Resource Languages in FW2

# Fineweb Edu for English

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm
import os
import pandas as pd

# Parameters
TARGET_TOKEN_COUNT = 1_000_000_000  # 1 billion tokens
BATCH_SIZE = 100_000
save_dir = "data/eng_Latn"
file_prefix = "eng_Latn"

# Setup
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
os.makedirs(save_dir, exist_ok=True)

print("LOADED TOKENIZER")

dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

print("Loaded DATASET")

# Streaming loop
buffer = []
total_tokens = 0
shard_id = 1
pbar = tqdm(total=TARGET_TOKEN_COUNT, unit="tokens")

for example in dataset:
    text = example.get("text", "").strip()
    if not text:
        continue

    input_ids = tokenizer(text, truncation=False, padding=False)["input_ids"]
    token_count = len(input_ids)

    buffer.append({"text": text})
    total_tokens += token_count
    pbar.update(token_count)

    if len(buffer) >= BATCH_SIZE or total_tokens >= TARGET_TOKEN_COUNT:
        # Save this shard to a parquet file
        df = pd.DataFrame(buffer)
        shard_name = f"{file_prefix}_{shard_id:04d}_fw2.parquet"
        df.to_parquet(os.path.join(save_dir, shard_name), index=False)
        shard_id += 1
        buffer = []

    if total_tokens >= TARGET_TOKEN_COUNT:
        break

# Save any remaining buffer
if buffer:
    df = pd.DataFrame(buffer)
    shard_name = f"{file_prefix}_{shard_id:04d}_fw2.parquet"
    df.to_parquet(os.path.join(save_dir, shard_name), index=False)

pbar.close()
print(f"\n✅ Done. Saved {shard_id} shards with ~{total_tokens:,} tokens to: {save_dir}")


# WURA

In [ ]:
# Create mapping of African language lable to Wura langauges
afri_wura_lang = {
    lang: flores_200_mapping[lang]
    for lang in african_language_list
    if lang in flores_200_mapping and flores_200_mapping[lang] in wura_langs
}

print(afri_wura_lang)
# print length of afri_wura_lang
print(f"Number of languages in afri_wura_lang: {len(afri_wura_lang)}")

In [ ]:
# High resource languages avaialbe in Wura

hr_wura_lang = {
    lang: flores_200_mapping[lang]
    for lang in high_lang_list
    if lang in flores_200_mapping and flores_200_mapping[lang] in wura_langs
}

print(hr_wura_lang)

## Downloading Wura for African Languages, Passage Level

In [ ]:
# Download African Languages for Wura

import os
from tqdm import tqdm

def download_wura_data(lang_code: str, save_dir: str):
    """
    Download the WURA dataset (PASSAGE LEVEL) and save it as a text file
    
    Args:
        lang_code: Language code for the dataset (e.g., 'hau', 'ibo', 'yor')
        save_dir: Directory to save the downloaded dataset
    """
    # Create the save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(save_dir, f"{lang_code}_wura.txt")
    
    # Load the dataset
    print(f"Loading WURA dataset for language: {lang_code}")
    data = load_dataset("castorini/wura", lang_code, level="passage", verification_mode="no_checks")
    
    # Write the text data to file
    print(f"Saving data to {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in tqdm(data['train']):
            f.write(item['text'] + '\n')
    
    print(f"Dataset saved to {output_file}")
    return output_file



for lang_code, wura_code in afri_wura_lang.items():
    save_dir = f"data/{lang_code}"
    download_wura_data(wura_code, save_dir)


## Downloading WURA for High Resource langauges, Passage Level

In [ ]:
# Download African Languages for Wura

import os
from tqdm import tqdm

def download_wura_data(lang_code: str, save_dir: str):
    """
    Download the WURA dataset (PASSAGE LEVEL) and save it as a text file
    
    Args:
        lang_code: Language code for the dataset (e.g., 'hau', 'ibo', 'yor')
        save_dir: Directory to save the downloaded dataset
    """
    # Create the save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(save_dir, f"{lang_code}_wura.txt")
    
    # Load the dataset
    print(f"Loading WURA dataset for language: {lang_code}")
    data = load_dataset("castorini/wura", lang_code, level="passage", verification_mode="no_checks")
    
    # Write the text data to file
    print(f"Saving data to {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in tqdm(data['train']):
            f.write(item['text'] + '\n')
    
    print(f"Dataset saved to {output_file}")
    return output_file



for lang_code, wura_code in hr_wura_lang.items():
    save_dir = f"data/{lang_code}"
    download_wura_data(wura_code, save_dir)


In [ ]:
data = load_dataset("castorini/wura", 'eng', level="passage", verification_mode="no_checks")
print(data['train'].column_names)
print(data['train'][0])

# Download for document level

In [ ]:
data = load_dataset("castorini/wura", "yor", level="document", verification_mode="no_checks")
                    
print(data['train'].column_names)
print(data['train'][1])

In [ ]:
data = load_dataset("castorini/wura", "yor", level="passage", verification_mode="no_checks")
                    
print(data['train'].column_names)

In [ ]:
#!/usr/bin/env python3
import os
import json
from datasets import load_dataset
from tqdm import tqdm

def download_wura_data(lang_code: str, save_dir: str):
    """
    Download the WURA dataset (Document LEVEL) and save it as a jsonl file,
    concatenating `headline` + `content` into `text`, and putting everything
    else under `hyperparam`.
    """
    os.makedirs(save_dir, exist_ok=True)
    output_file = os.path.join(save_dir, f"{lang_code}_wura_documentLevel.jsonl")

    print(f"Loading WURA dataset for language: {lang_code}")
    data = load_dataset(
        "castorini/wura",
        lang_code,
        level="document",
        verification_mode="no_checks"
    )

    print(f"Writing transformed JSONL to {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in tqdm(data['train'], desc="items"):
            # 1) safe-get + strip headline/content
            headline = (item.get('headline') or "").strip()
            content  = (item.get('content')  or "").strip()

            # 2) concatenate and trim any extra whitespace
            text = f"{headline} {content}".strip()

            # 3) stash the rest under hyperparam
            hyperparam = {
                k: v
                for k, v in item.items()
                if k not in ('headline', 'content')
            }

            # 4) write one JSON line
            out = {
                "text": text,
                "hyperparam": hyperparam
            }
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Done → {output_file}")
    return output_file

if __name__ == "__main__":
    for lang_folder, wura_code in afri_wura_lang.items():
        save_dir = os.path.join("data", lang_folder)
        download_wura_data(wura_code, save_dir)


In [ ]:
#!/usr/bin/env python3
import os
import json
from datasets import load_dataset
from tqdm import tqdm

def download_wura_data(lang_code: str, save_dir: str):
    """
    Download the WURA dataset (Document LEVEL) and save it as a jsonl file,
    concatenating `headline` + `content` into `text`, and putting everything
    else under `hyperparam`.
    """
    os.makedirs(save_dir, exist_ok=True)
    output_file = os.path.join(save_dir, f"{lang_code}_wura_documentLevel.jsonl")

    print(f"Loading WURA dataset for language: {lang_code}")
    data = load_dataset(
        "castorini/wura",
        lang_code,
        level="document",
        verification_mode="no_checks"
    )

    print(f"Writing transformed JSONL to {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in tqdm(data['train'], desc="items"):
            # 1) safe-get + strip headline/content
            headline = (item.get('headline') or "").strip()
            content  = (item.get('content')  or "").strip()

            # 2) concatenate and trim any extra whitespace
            text = f"{headline} {content}".strip()

            # 3) stash the rest under hyperparam
            hyperparam = {
                k: v
                for k, v in item.items()
                if k not in ('headline', 'content')
            }

            # 4) write one JSON line
            out = {
                "text": text,
                "hyperparam": hyperparam
            }
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Done → {output_file}")
    return output_file

if __name__ == "__main__":
    for lang_folder, wura_code in hr_wura_lang.items():
        save_dir = os.path.join("data", lang_folder)
        download_wura_data(wura_code, save_dir)
# Download Wura dataset for high resource languages

In [ ]:
# load your single JSONL as the “train” split
data = load_dataset(
    "json",
    data_files={ "train": "data/<lang_code>/<lang_code>_wura_documentLevel.jsonl" }
)

# inspect the columns
print(data["train"].column_names)
print(data["train"][0])

# Madlad 400

## Madlad 400 for African Languages

In [ ]:
# Defining the madlad 400 mapping

afri_madlad_langs = {
    "afr_Latn": "af",
    "aka_Latn": "ak",
    "amh_Ethi": "am",
    "bam_Latn": "bm",
    "dik_Latn": "din",
    "dyu_Latn": "dyu",
    "ewe_Latn": "ee",
    "fon_Latn": "fon",
    "fuv_Latn": "ff",
    "gaz_Latn": "om",   
    "hau_Latn": "ha",  
    "ibo_Latn": "ig",   
    "kbp_Latn": "kbp",
    "kin_Latn": "rw",
    "kmb_Latn": "kmb",
    "kon_Latn": "kg",
    "lin_Latn": "ln",
    "lug_Latn": "lg",
    "run_Latn": "rn",
    "sag_Latn": "sg",
    "sna_Latn": "sn",
    "som_Latn": "so",
    "sot_Latn": "st",
    "ssw_Latn": "ss",
    "swh_Latn": "sw",
    "tir_Ethi": "ti",
    "tsn_Latn": "tn",
    "tso_Latn": "ts",
    "tzm_Tfng": "ber",
    "wol_Latn": "wo",
    "xho_Latn": "xh",
    "yor_Latn": "yo",
    "zul_Latn": "zu"
}

hr_madlad_langs = {
    "eng_Latn": "en",
    "fra_Latn": "fr",
    "por_Latn": "pt",
    "arb_Arab": "ar"
}

In [ ]:
import os
import requests
import gzip
import shutil

def download_and_merge_madlad_clean_files(lang_code: str, output_filename: str, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)

    base_url = f"https://huggingface.co/datasets/allenai/madlad-400/resolve/main/data/{lang_code}/"
    i = 0
    temp_dir = os.path.join(save_dir, "temp_chunks")
    os.makedirs(temp_dir, exist_ok=True)

    # Step 1: Download all chunks
    print(f"Downloading all chunks for {lang_code} ...")
    while True:
        filename = f"{lang_code}_clean_{i:04d}.jsonl.gz"
        url = base_url + filename
        local_path = os.path.join(temp_dir, filename)

        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded: {filename}")
            i += 1
        else:
            print(f"No more files after: {filename}")
            break

    # Step 2: Decompress and merge into single file
    final_path = os.path.join(save_dir, output_filename)
    with open(final_path, 'wb') as outfile:
        for j in range(i):
            part_file = os.path.join(temp_dir, f"{lang_code}_clean_{j:04d}.jsonl.gz")
            with gzip.open(part_file, 'rb') as f_in:
                shutil.copyfileobj(f_in, outfile)

    print(f"\nFinal merged file saved to: {final_path}")

    # Step 3: Clean up
    shutil.rmtree(temp_dir)
    print("🧹 Cleaned up temp files.")


for lang_code, madlad_code in afri_madlad_langs.items():
    download_and_merge_madlad_clean_files(madlad_code, f"{lang_code}_ml400.jsonl", f"data/{lang_code}")


In [ ]:
import json

# Path to one of your merged JSONL files
file_path = "data/<lang_code>/<lang_code>_ml400.jsonl"

# Read the first non-empty line
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            sample = json.loads(line)
            break

# Print the keys (columns)
print("Columns:", list(sample.keys()))

In [ ]:
import os
import requests
import gzip
import shutil

def download_and_merge_madlad_clean_files(lang_code: str, output_filename: str, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)

    base_url = f"https://huggingface.co/datasets/allenai/madlad-400/resolve/main/data/{lang_code}/"
    i = 0
    temp_dir = os.path.join(save_dir, "temp_chunks")
    os.makedirs(temp_dir, exist_ok=True)

    # Step 1: Download all chunks
    print(f"Downloading all chunks for {lang_code} ...")
    while True:
        filename = f"{lang_code}_clean_{i:04d}.jsonl.gz"
        url = base_url + filename
        local_path = os.path.join(temp_dir, filename)

        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded: {filename}")
            i += 1
        else:
            print(f"No more files after: {filename}")
            break

    # Step 2: Decompress and merge into single file
    final_path = os.path.join(save_dir, output_filename)
    with open(final_path, 'wb') as outfile:
        for j in range(i):
            part_file = os.path.join(temp_dir, f"{lang_code}_clean_{j:04d}.jsonl.gz")
            with gzip.open(part_file, 'rb') as f_in:
                shutil.copyfileobj(f_in, outfile)

    print(f"\nFinal merged file saved to: {final_path}")

    # Step 3: Clean up
    shutil.rmtree(temp_dir)
    print("🧹 Cleaned up temp files.")


for lang_code, madlad_code in hr_madlad_langs.items():
    download_and_merge_madlad_clean_files(madlad_code, f"{lang_code}_ml400.jsonl", f"data/{lang_code}")


# Extra Data

## Tswana

In [ ]:
# Output folder & path
output_dir = "./data/tsn_Latn"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "tsn_Latn_extra.jsonl")

# Load the extra dataset
ds = load_dataset("OxxoCodes/Marothodi", split="train")

# Print structure of first example to verify
if len(ds) > 0:
    print("Marothodi example structure:", ds[0])

# Check if the file exists and how many lines it has
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        existing_lines = sum(1 for _ in f)

    if existing_lines >= len(ds):
        print(f"{output_path} already contains all {existing_lines} examples, skipping.")
    else:
        print(f"{output_path} has {existing_lines}/{len(ds)} examples — appending missing...")
        with open(output_path, "a", encoding="utf-8") as out_f:
            for i, example in enumerate(ds):
                if i < existing_lines:
                    continue
                
                # Create standardized format with "text" field
                if "text" in example:
                    json.dump({"text": example["text"]}, out_f)
                elif "sentence" in example:  # Assuming the dataset might have a "sentence" field
                    json.dump({"text": example["sentence"]}, out_f)
                else:
                    # If no clear text field, identify the appropriate field based on dataset structure
                    # For example, concatenate multiple fields or use a specific field:
                    # You may need to adjust this based on the actual structure
                    content = str(example)
                    json.dump({"text": content}, out_f)
                
                out_f.write("\n")
        
        print(f"Appended {len(ds) - existing_lines} new examples to {output_path}")
else:
    # File does not exist; write all from scratch
    with open(output_path, "w", encoding="utf-8") as out_f:
        for example in ds:
            # Create standardized format with "text" field
            if "text" in example:
                json.dump({"text": example["text"]}, out_f)
            elif "sentence" in example:  # Assuming the dataset might have a "sentence" field
                json.dump({"text": example["sentence"]}, out_f)
            else:
                # If no clear text field, identify the appropriate field based on dataset structure
                content = str(example)
                json.dump({"text": content}, out_f)
            
            out_f.write("\n")
    
    print(f"Wrote all {len(ds)} examples to {output_path}")

In [ ]:
ds = load_dataset("OxxoCodes/Marothodi", split="train")
print(ds.column_names)
print(ds[0])

# Convert to LLaMMa Factory Style

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class DatasetTransformer:
    def __init__(self, root_dir="data", output_dir="preprocessed_data", chunk_size=50_000):
        self.root_dir = Path(root_dir)
        self.output_dir = Path(output_dir)
        self.chunk_size = chunk_size
        self.output_dir.mkdir(exist_ok=True, parents=True)
        
    def scan_directory(self):
        """Scan the data directory for language subdirectories and files."""
        lang_dirs = [d for d in self.root_dir.iterdir() if d.is_dir()]
        logger.info(f"Found {len(lang_dirs)} language directories")
        return lang_dirs
    
    def process_all(self):
        """Process all language directories and their files."""
        lang_dirs = self.scan_directory()
        
        for lang_dir in lang_dirs:
            lang_code = lang_dir.name
            logger.info(f"Processing language: {lang_code}")
            
            # Create output directory for this language
            lang_output_dir = self.output_dir / lang_code
            lang_output_dir.mkdir(exist_ok=True)
            
            # Process each file in the language directory
            for file_path in lang_dir.glob("*"):
                if file_path.is_file():
                    self.process_file(file_path, lang_output_dir, lang_code)
    
    def process_file(self, file_path, output_dir, lang_code):
        """Process a single file based on its type and name pattern."""
        filename = file_path.name
        logger.info(f"Processing file: {filename}")
        output_path = output_dir / f"{file_path.stem}.parquet"

        # ←— Skip if we already wrote this file
        if output_path.exists():
            logger.info(f"Skipping {filename}: output already at {output_path}")
            return
        
        # Determine file type
        if filename.endswith(".jsonl") and "wura" in filename: # This will be the document level of wura
            self.process_wura_jsonl(file_path, lang_code, output_path)
            return
        if filename.endswith(".parquet") and "fw2" in filename:
            self.process_fw2(file_path, lang_code, output_path)
            logger.info(f"Chunk-wrote FW2 to {output_path}")
            return
        elif filename.endswith(".txt") and "wura" in filename: # This will be the passage level of wura
            # pass output_path so it can write itself
            self.process_wura(file_path, lang_code, output_path)
            logger.info(f"Chunk-wrote Wura → {output_path}")
            return
        elif filename.endswith(".jsonl") and "ml400" in filename:
            self.process_madlad400(file_path, lang_code, output_path)
            logger.info(f"Chunk-wrote MADLAD400 → {output_path}")
            return
        elif filename.endswith(".jsonl") and "extra" in filename:
            df = self.process_extra_data(file_path, lang_code)
        else:
            logger.warning(f"Unknown file format for {filename}, skipping")
            return
        
        # Save the transformed dataframe
        output_filename = f"{file_path.stem}.parquet"
        output_path = output_dir / output_filename
        df.to_parquet(output_path, index=False)
        logger.info(f"Saved transformed data to {output_path}")

    def process_wura_jsonl(self, file_path: Path, lang_code: str, output_path: Path):
        """Stream Wura JSONL in chunks and write to Parquet."""
        writer = None
        batch, n = [], 0
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    entry = json.loads(line)
                    batch.append({
                        'text': entry.get('text', ''),
                        'hyperparam': {
                            'id':       f"wura_{lang_code}_{i}",
                            'language': lang_code,
                            'dataset':  'wura'
                        }
                    })
                    n += 1
                    if n >= self.chunk_size:
                        tbl = pa.Table.from_pandas(
                            pd.DataFrame(batch)[['text','hyperparam']],
                            preserve_index=False
                        )
                        if writer is None:
                            writer = pq.ParquetWriter(str(output_path), schema=tbl.schema)
                        writer.write_table(tbl)
                        batch, n = [], 0
        except Exception as e:
            logger.error(f"Error streaming Wura JSONL {file_path}: {e}")

        # flush remainder
        if batch:
            tbl = pa.Table.from_pandas(
                pd.DataFrame(batch)[['text','hyperparam']],
                preserve_index=False
            )
            if writer is None:
                writer = pq.ParquetWriter(str(output_path), schema=tbl.schema)
            writer.write_table(tbl)

        # if nothing written, stub out an empty file
        if writer:
            writer.close()
        else:
            empty = pa.Table.from_pandas(
                pd.DataFrame(columns=['text','hyperparam']),
                preserve_index=False
            )
            pq.write_table(empty, str(output_path))
        logger.info(f"Wrote Wura JSONL → {output_path}")
    
    def process_fw2(self, file_path, lang_code, output_path):
        """
        Process one large FW2 parquet by row-group, build hyperparam, and stream directly
        to output_path (a pathlib.Path to the final .parquet).
        """
        # List every field you *want*, but we'll only read the ones that actually exist
        wanted = [
            'text','id','dump','url','date','file_path',
            'language','language_score','language_script',
            'minhash_cluster_size','top_langs'
        ]

        # Open the source Parquet
        pqf = pq.ParquetFile(str(file_path))
        writer = None

        for rg in range(pqf.num_row_groups):
            # 1) Figure out which of our "wanted" fields are actually in this file
            existing = set(pqf.schema.names)
            to_read  = [c for c in wanted if c in existing]

            # always require "text" so we can output something
            if 'text' not in to_read:
                logger.warning(f"No 'text' column in row-group {rg}, skipping")
                continue

            # 2) Read just those columns
            table = pqf.read_row_group(rg, columns=to_read)
            df    = table.to_pandas()

            # 3) For any field we *wanted* but wasn’t present, add a default
            for c in wanted:
                if c not in df.columns:
                    df[c] = None

            # 4) Build hyperparam *safely* using row.get() with defaults
            def make_hp(row):
                return {
                    'id':                    row.get('id', '')                   or '',
                    'dump':                  row.get('dump', '')                 or '',
                    'url':                   row.get('url', '')                  or '',
                    'date':                  str(row.get('date', ''))            or '',
                    'file_path':             row.get('file_path', '')            or '',
                    'language':              lang_code,
                    'language_score':        row.get('language_score', 0.0)      or 0.0,
                    'language_script':       row.get('language_script', '')      or '',
                    'minhash_cluster_size':  row.get('minhash_cluster_size', 0) or 0,
                    'top_langs':             row.get('top_langs', [])            or [],
                    'dataset':               'fw2'
                }

            df['hyperparam'] = df.apply(make_hp, axis=1)

            # 5) Stream it straight out to Parquet without concatenating in RAM
            out_table = pa.Table.from_pandas(
                df[['text','hyperparam']],
                preserve_index=False
            )

            if writer is None:
                writer = pq.ParquetWriter(str(output_path), schema=out_table.schema)
            writer.write_table(out_table)

            # free up memory
            del df, table, out_table

        # 6) Clean up
        if writer:
            writer.close()
        else:
            # no data at all? write an empty stub
            empty = pa.Table.from_pandas(
                pd.DataFrame(columns=['text','hyperparam']),
                preserve_index=False
            )
            pq.write_table(empty, str(output_path))

    
    def process_wura(self, file_path, lang_code, output_path, chunk_size=50_000):
        """Process Wura `.txt` by streaming in chunks and writing to Parquet."""
        writer = None
        rows = []
        count = 0

        # 1) Stream the file line by line
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                line = line.rstrip('\n')
                # parse id/text
                if '\t' in line:
                    doc_id, text = line.split('\t', 1)
                else:
                    doc_id = f"wura_{lang_code}_{i}"
                    text   = line

                rows.append({
                    'text': text,
                    'hyperparam': {
                        'id':       doc_id,
                        'language': lang_code,
                        'dataset':  'wura'
                    }
                })
                count += 1

                # 2) Once we hit chunk_size, flush to Parquet
                if count >= chunk_size:
                    df = pd.DataFrame(rows)
                    table = pa.Table.from_pandas(df[['text','hyperparam']],
                                                preserve_index=False)
                    if writer is None:
                        writer = pq.ParquetWriter(str(output_path),
                                                schema=table.schema)
                    writer.write_table(table)

                    # reset
                    rows = []
                    count = 0

        # 3) Flush any remaining rows
        if rows:
            df = pd.DataFrame(rows)
            table = pa.Table.from_pandas(df[['text','hyperparam']],
                                        preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(output_path),
                                        schema=table.schema)
            writer.write_table(table)

        # 4) If we never saw any data, write an empty stub
        if writer:
            writer.close()
        else:
            empty = pa.Table.from_pandas(
                pd.DataFrame(columns=['text','hyperparam']),
                preserve_index=False
            )
            pq.write_table(empty, str(output_path))

        logger.info(f"Wrote Wura data for {lang_code}: {output_path}")
        
    def process_madlad400(self, file_path, lang_code, output_path, chunk_size=50_000):
        """Process MADLAD400 JSONL by streaming in chunks and writing to Parquet."""
        writer = None
        batch = []
        n = 0

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    try:
                        entry = json.loads(line)
                    except json.JSONDecodeError:
                        logger.warning(f"Invalid JSON on line {i} in {file_path}")
                        continue

                    text = entry.get('text', '')
                    batch.append({
                        'text': text,
                        'hyperparam': {
                            'id':       f"madlad400_{lang_code}_{i}",
                            'language': lang_code,
                            'dataset':  'madlad400'
                        }
                    })
                    n += 1

                    # flush chunk
                    if n >= chunk_size:
                        tbl = pa.Table.from_pandas(
                            pd.DataFrame(batch)[['text','hyperparam']],
                            preserve_index=False
                        )
                        if writer is None:
                            writer = pq.ParquetWriter(str(output_path), schema=tbl.schema)
                        writer.write_table(tbl)
                        batch, n = [], 0

        except Exception as e:
            logger.error(f"Error streaming MADLAD400 {file_path}: {e}")

        # flush remainder
        if batch:
            tbl = pa.Table.from_pandas(
                pd.DataFrame(batch)[['text','hyperparam']],
                preserve_index=False
            )
            if writer is None:
                writer = pq.ParquetWriter(str(output_path), schema=tbl.schema)
            writer.write_table(tbl)

        # if nothing written, stub out an empty file
        if writer:
            writer.close()
        else:
            empty = pa.Table.from_pandas(
                pd.DataFrame(columns=['text','hyperparam']),
                preserve_index=False
            )
            pq.write_table(empty, str(output_path))

        logger.info(f"Wrote MADLAD400 for {lang_code}: {output_path}")
    
    def process_extra_data(self, file_path, lang_code):
        """Process extra data JSONL files."""
        try:
            data = []
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    try:
                        # Parse the JSON line
                        entry = json.loads(line)
                        text = entry.get('text', '')
                        source = entry.get('source', '')
                        source_category = entry.get('source-category', '')
                        
                        data.append({
                            'text': text,
                            'hyperparam': {
                                'id': f"extra_{lang_code}_{i}",
                                'language': lang_code,
                                'source': source,
                                'source_category': source_category,
                                'dataset': 'extra'
                            }
                        })
                    except json.JSONDecodeError:
                        logger.warning(f"Invalid JSON on line {i} in file {file_path}")
                    except Exception as e:
                        logger.warning(f"Error processing line {i} in extra data file: {e}")
            
            return pd.DataFrame(data)
        except Exception as e:
            logger.error(f"Error processing extra data file {file_path}: {e}")
            return pd.DataFrame(columns=['text', 'hyperparam'])


def main():
    """Main function to run the dataset transformer."""
    # Configure these paths as needed
    input_dir = "data"
    output_dir = "preprocessed_data"
    
    transformer = DatasetTransformer(root_dir=input_dir, output_dir=output_dir)
    transformer.process_all()
    logger.info("Dataset transformation complete!")


if __name__ == "__main__":
    main()

In [ ]:
dataset = load_dataset(
    "parquet",
    data_files={"preprocessed_data/<lang_code>/<file>.parquet"},
    split="train",
    streaming=True
)

print("Available columns:", dataset.features)
print("First example:", next(iter(dataset)))


# Create My Huggingface Dataset

In [ ]:
# Download dataset

from datasets import load_dataset, get_dataset_config_names
configs = get_dataset_config_names("<YourOrganization>/<YourRepo>")

print("Available languages:", configs)


ds_aeb = load_dataset("<YourOrganization>/<YourRepo>", "aeb_Arab")

print(ds_aeb)

print(ds_aeb["fineweb2"][0])        # first record from the fineweb2 split

# Command for Dataset Loading

In [ ]:
# Loading a single language

from datasets import load_dataset, get_dataset_config_names
configs = get_dataset_config_names("<YourOrganization>/<YourRepo>")

print("Available languages:", configs)

ds_aeb = load_dataset("<YourOrganization>/<YourRepo>", "amh_Ethi")

print(ds_aeb)

print(ds_aeb["wura_documentLevel"][0])        # first record from the WURA document-level split

print(ds_aeb["fineweb2"][0])        # first record from the fineweb2 split

In [ ]:
# Load a single langauge with specified splits

from datasets import load_dataset

# only load the fineweb2 split for English-Latin
fw2_eng = load_dataset(
    "<YourOrganization>/<YourRepo>",
    name="eng_Latn",
    split="fineweb2"
)
print(fw2_eng)

In [ ]:
# Load multiple languages (all splits) in a loop

from datasets import load_dataset

langs = ["eng_Latn", "fra_Latn", "por_Latn"]
all_ds = {}

for lang in langs:
    all_ds[lang] = load_dataset("<YourOrganization>/<YourRepo>", name=lang)

# now all_ds["fra_Latn"]["wura_documentLevel"] etc. are available


## Creating the Dataset_Type hyperparam 

This is to keep track and load exactly the dataset type I want in the future

In [ ]:
#!/usr/bin/env python3
import logging
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

def infer_origin(stem: str) -> str | None:
    stem = stem.lower()
    if "fw2" in stem:
        return "fw2"
    if "documentlevel" in stem:
        return "wuraDocumentLevel"
    if "fwedu" in stem:
        return "fwedu"
    if "ml400" in stem:
        return "madlad400"
    if "wura" in stem:
        return "wura"
    if "extra" in stem:
        return "extra"
    return None

def process_file_in_chunks(pq_path: Path, origin: str, batch_size: int = 50_000):
    """
    Read pq_path in batches, add a constant `dataset_origin` column,
    and overwrite the file in a memory-safe way.
    """
    # Open source file
    parquet_file = pq.ParquetFile(pq_path)

    # Extend schema with new string column
    new_schema = parquet_file.schema_arrow.append(
        pa.field("dataset_origin", pa.string())
    )

    tmp_path = pq_path.with_suffix(".tmp.parquet")
    writer = pq.ParquetWriter(tmp_path, new_schema, compression="snappy")

    # Stream through row‐groups / batches
    for batch in parquet_file.iter_batches(batch_size=batch_size):
        table = pa.Table.from_batches([batch], schema=parquet_file.schema_arrow)
        # constant column with `origin`
        origin_col = pa.array([origin] * table.num_rows)
        table = table.append_column("dataset_origin", origin_col)
        writer.write_table(table)

    writer.close()
    tmp_path.replace(pq_path)


def add_origin_column(root_dir: str = "preprocessed_data"):
    root = Path(root_dir)
    if not root.is_dir():
        logger.error(f"Root directory {root_dir!r} does not exist or is not a folder.")
        return

    for lang_dir in root.iterdir():
        if not lang_dir.is_dir():
            continue

        logger.info(f"Scanning language folder: {lang_dir.name}")
        for pq_file in lang_dir.glob("*.parquet"):
            # Detect existing column
            try:
                schema = pq.ParquetFile(pq_file).schema_arrow
                if "dataset_origin" in schema.names:
                    logger.info(f"{pq_file.name} already has dataset_origin; skipping")
                    continue
            except Exception as e:
                logger.warning(f"Could not read schema for {pq_file.name}: {e}")
                continue

            origin = infer_origin(pq_file.stem)
            if origin is None:
                logger.warning(f"Could not infer origin for {pq_file.name}; skipping")
                continue

            logger.info(f"Processing {pq_file.name} in chunks (origin={origin})")
            try:
                process_file_in_chunks(pq_file, origin)
                logger.info(f"✅ Updated {pq_file.name}")
            except Exception as e:
                logger.error(f"Failed to process {pq_file.name}: {e}")

if __name__ == '__main__':
    add_origin_column("preprocessed_data/<lang_code>")


In [ ]:
dataset = load_dataset(
    "parquet",
    data_files={"preprocessed_data/<lang_code>/<file>.parquet"},
    split="train",
    streaming=True
)

print("Available columns:", dataset.features)

print(next(iter(dataset)))
